# MitoSAM demo — from image to quantified biology in one notebook

Walkthrough of the full pipeline on the synthetic benchmark (no GPU, no downloads):

1. **Data** — generate a cohort with known ground truth
2. **Segmentation** — classical baseline vs promptable MitoSAM
3. **Features** — morphology / texture / spatial, per organelle and per FOV
4. **Classification** — ablation over feature families with honest metrics
5. **Explainability** — SHAP: which features carry the decision
6. **Interface** — the same numbers in the Gradio demo

Every figure produced here also exists as a file in `results/figures/` via `pipelines/run_all.py`.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'MitoSAM':
    sys.path.insert(0, str(REPO.parent))
    %cd $REPO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MitoSAM.config import MitoSAMConfig
from MitoSAM.utils.io import load_image_gray, load_mask, list_images, find_mask_for_image
from MitoSAM.utils.viz import palette
print('MitoSAM ready')

## 1 · Data: a synthetic cohort with known ground truth

In [ ]:
from MitoSAM.data.synthetic import create_cohort

cohort = create_cohort('examples/synthetic', n_per_class=12, seed=42)
labels = pd.read_csv(cohort / 'labels.csv')
print(labels.group.value_counts())

fig, axes = plt.subplots(2, 3, figsize=(10, 6.6))
for row, group in enumerate(['control', 'treated']):
    stems = labels.loc[labels.group == group, 'image_id'].head(3)
    for col, stem in enumerate(stems):
        img = load_image_gray(cohort / 'images' / f'{stem}.png')
        msk = load_mask(cohort / 'masks' / f'{stem}_mask.png')
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].contour(msk, levels=[0.5], colors=[palette('secondary')], linewidths=0.6)
        axes[row, col].set_title(f'{group} / {stem}', fontsize=9)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
fig.suptitle('Synthetic SR-like fields (orange = GT outline)', fontsize=11)
fig.tight_layout()

## 2 · Segmentation: why a threshold is not enough

The panel compares Otsu with GT on a real 16-bit FOV **and** on the synthetic data —
the same code path, very different outcome.

In [ ]:
from skimage.filters import threshold_otsu
from MitoSAM.utils.metrics import dice_score

# real 16-bit FOV (if the private dataset is mounted)
real_img_dir, real_mask_dir = Path('W:/mitochondria_dataset/train/images'), Path('W:/mitochondria_dataset/train/masks')
fig, axes = plt.subplots(2, 3, figsize=(10, 6.6))
for row, (img_dir, mask_dir, tag) in enumerate([(real_img_dir, real_mask_dir, 'real 16-bit'),
                                                (cohort/'images', cohort/'masks', 'synthetic')]):
    files = list_images(img_dir)[:1] if img_dir.exists() else []
    if not files:
        axes[row,0].axis(False); axes[row,0].text(.5,.5,f'{tag}: not available', ha='center'); continue
    img = load_image_gray(files[0])
    gt = load_mask(find_mask_for_image(files[0], mask_dir))
    otsu = img > threshold_otsu(img)
    axes[row,0].imshow(img, cmap='gray'); axes[row,0].set_title(f'{tag}: input')
    axes[row,1].imshow(otsu, cmap='gray'); axes[row,1].set_title(f'Otsu (Dice={dice_score(otsu, gt):.3f})')
    axes[row,2].imshow(gt, cmap='gray'); axes[row,2].set_title('ground truth')
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()

In [ ]:
# Promptable MitoSAM (zero-shot SAM backbone if no fine-tuned checkpoint exists)
try:
    from MitoSAM.models.mitosam import MitoSAM

    model = MitoSAM('vit_b', checkpoint='weights/sam_vit_b_01ec64.pth')
    img = load_image_gray(cohort/'images'/'control_000.png')
    model.set_image(img)
    items = model.predict_dense(grid_size=24, score_threshold=0.65,
                                stability_threshold=0.75, min_area=30)
    union = np.zeros(img.shape, bool)
    for it in items: union |= it['segmentation']
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
    ax[0].imshow(img, cmap='gray'); ax[0].set_title('input')
    ax[1].imshow(img, cmap='gray', alpha=.55)
    for it in items[:40]:
        ax[1].contour(it['segmentation'], levels=[.5], colors=[palette('primary')], linewidths=.4)
    ax[1].set_title(f'MitoSAM zero-shot, {len(items)} masks')
    for a in ax: a.set_xticks([]); a.set_yticks([])
except ImportError as e:
    print('SAM/torch not installed in this env -> skipping interactive demo:', e)

## 3 · Features: 130 numbers per field of view

In [ ]:
from MitoSAM.features.extract import build_feature_tables

cfg = MitoSAMConfig()
cfg.data.image_dir = str(cohort/'images')
cfg.data.mask_dir = str(cohort/'masks')
labels_map = dict(zip(labels.image_id.astype(str), labels.group))
per_image, per_organelle = build_feature_tables(cfg, 'results/tables', labels_map)
print('per-FOV table:', per_image.shape, '| per-organelle table:', per_organelle.shape)
per_image.head(3)

In [ ]:
key = ['n_mitochondria','area_mean','aspect_ratio_mean','circularity_mean',
       'intensity_entropy','LBP_entropy','nn_mean']
summary = per_image.groupby('group')[key].mean()
fig, ax = plt.subplots(figsize=(8.4, 3.6))
x = np.arange(len(key)); w = 0.38
ax.bar(x - w/2, summary.loc['control'], w, label='control', color=palette('primary'))
ax.bar(x + w/2, summary.loc['treated'], w, label='treated', color=palette('secondary'))
ax.set_xticks(x); ax.set_xticklabels([k.replace('_','\n') for k in key], fontsize=8)
ax.set_ylabel('mean per FOV'); ax.legend(frameon=False)
ax.set_title('Group signatures across the three feature families', fontsize=11)
plt.tight_layout()

## 4 · Classification with ablation + permutation control

In [ ]:
from MitoSAM.evaluation.classification import run_classification
from MitoSAM.evaluation.permutation_test import permutation_control

payload = run_classification(cfg, 'results/tables/features_per_image.csv', 'results')
null = permutation_control(cfg, 'results/tables/features_per_image.csv', n_permutations=10, out_dir='results')
pd.DataFrame(payload['results'])[['model','feature_subset','test_auc','cv_auc_mean','generalization_gap']].head(12)

In [ ]:
ablation = pd.read_csv('results/tables/ablation_summary.csv')
fig, ax = plt.subplots(figsize=(7.6, 3.2))
ax.barh(ablation.feature_subset, ablation.rcv_auc_mean,
        xerr=ablation.rcv_auc_std, color=palette('primary'), capsize=2)
ax.axvline(null['observed_auc'], color=palette('secondary'), lw=2, label='best model')
ax.axvline(0.5, color='#BBBBBB', ls=':', label='chance')
ax.set_xlabel('repeated-CV AUC (mean ± std over 50 folds)'); ax.legend(frameon=False)
ax.set_title(f"Permutation null: {null['null_mean']:.2f} ± {null['null_std']:.2f} "
             f"(observed {null['observed_auc']:.2f})", fontsize=10)
plt.tight_layout()

## 5 · SHAP: which features carry the decision

In [ ]:
from MitoSAM.evaluation.shap_explain import compute_shap, plot_shap

shap_res = compute_shap(cfg, 'results/tables/features_per_image.csv')
plot_shap(shap_res, 'results/figures')
print('family shares:', shap_res['family_share'])
shap_res['mean_abs'].head(10)

## 6 · The same analysis as a product

* CLI: `python scripts/batch_process.py --input_dir ... --output_dir ... --features`
* Web UI: `python -m MitoSAM.gui.app` → http://127.0.0.1:7860
* One-command reproduction: `python -m MitoSAM.pipelines.run_all --help`

See `docs/TECHNICAL_REPORT.md` for the written narrative, limitations and roadmap.